In [20]:
from dataclasses import dataclass
from pathlib import Path

In [21]:


@dataclass
class DataModelConfig:
  root_dir: Path
  train_path: Path
  test_path: Path
  model_name: str
  alpha: float
  l1_ratio: float
  target_column: str

In [22]:
from datascience.constants import *
from datascience.utils.common import *

In [23]:
class ConfigurationManager:
    def __init__(self,config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_training(self):

        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        data_model_config = DataModelConfig(
                  root_dir =  config.root_dir,
                  train_path = config.train_path,
                  test_path = config.test_path,
                  model_name =  config.model_name,
                  alpha = params.alpha,
                  l1_ratio = params.l1_ratio,
                  target_name = schema.name)

        return data_model_config

In [24]:
import pandas as pd
import os
from datascience import logger
from sklearn.linear_model import ElasticNet
import joblib

In [25]:
class ModelTrainer:
    def __init__(self,config: DataModelConfig):
        self.config = config

    def train(self):
        train_data = pd.read_csv(self.config.train_path)
        test_data = pd.read_csv(self.config.test_path)


        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]


        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(train_x, train_y)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))


In [26]:
try:
    config=ConfigurationManager()
    data_model_config=config.get_model_training()
    data_model=ModelTrainer(config=data_model_config)
    data_model.train()
except Exception as e:
    raise e

2026-04-30 16:53:36,051: - INFO: - common - yaml file: /Users/benjaminbrooke/PycharmProjects/MLOps/Big_project/config/config.yaml loaded successfully
2026-04-30 16:53:36,052: - INFO: - common - yaml file: /Users/benjaminbrooke/PycharmProjects/MLOps/Big_project/params.yaml loaded successfully
2026-04-30 16:53:36,054: - INFO: - common - yaml file: /Users/benjaminbrooke/PycharmProjects/MLOps/Big_project/schema.yaml loaded successfully
2026-04-30 16:53:36,054: - INFO: - common - created directory at: artifacts
2026-04-30 16:53:36,054: - INFO: - common - created directory at: artifacts/model_trainer


TypeError: DataModelConfig.__init__() got an unexpected keyword argument 'target_name'